# Transformações

**Módulo:** 05 – Trigonometria (6º notebook do módulo)

**Pré-requisitos:** [`0502a_funcoes_circulares.ipynb`](0502a_funcoes_circulares.ipynb) (definição das seis funções circulares), [`0503a_relacoes_fundamentais.ipynb`](0503a_relacoes_fundamentais.ipynb) (relação fundamental, usada para conferir as fórmulas deduzidas), [`0504a_reducao_ao_1o_quadrante.ipynb`](0504a_reducao_ao_1o_quadrante.ipynb) (paridade das funções circulares, usada na dedução das fórmulas de subtração) e [`0505a_arcos_notaveis.ipynb`](0505a_arcos_notaveis.ipynb) (este notebook resolve diretamente a pergunta em aberto deixada ali)

**Objetivos de aprendizagem:**
- Deduzir as fórmulas de adição e subtração de arcos (sen, cos, tg) a partir do ciclo trigonométrico, e usá-las para calcular exatamente arcos combinados como 15°, 75° e 105°.
- Obter as fórmulas de arco duplo e arco triplo a partir das fórmulas de adição.
- Deduzir as fórmulas de arco metade — incluindo a tangente do arco metade — a partir das fórmulas de arco duplo.
- Transformar somas e diferenças de funções circulares em produtos, e reconhecer essa transformação em fenômenos físicos como o batimento sonoro.

**Tempo estimado:** 90 a 110 minutos

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/05_trigonometria/0506a_transformacoes.ipynb)

No notebook anterior, terminamos com uma pergunta em aberto: sen(45° − 30°) não é igual a sen(45°) − sen(30°) — então qual é a fórmula certa para calcular sen(a − b)? Neste notebook você vai deduzir essa fórmula (e várias outras) e finalmente descobrir o valor exato de sen(15°).

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
import ipywidgets as widgets
from IPython.display import display

# Paleta de cores Nord
NORD_FUNDO = "#2E3440"
NORD_PAINEL = "#3B4252"
NORD_LINHA = "#4C566A"
NORD_TEXTO = "#ECEFF4"
NORD_TEXTO_SEC = "#D8DEE9"
COR_PONTO = "#88C0D0"
COR_DESTAQUE = "#A3BE8C"
COR_ALERTA = "#BF616A"
COR_SECUNDARIA = "#81A1C1"
COR_EXTRA = "#B48EAD"
COR_AVISO = "#EBCB8B"


def estilizar_eixo(ax) -> None:
    """Aplica o estilo Nord padrão a um eixo do matplotlib."""
    ax.set_facecolor(NORD_FUNDO)
    ax.tick_params(colors=NORD_TEXTO_SEC)
    for spine in ax.spines.values():
        spine.set_color(NORD_LINHA)
    ax.grid(color=NORD_LINHA, linewidth=0.5, alpha=0.5)

## 1. Um Problema em Aberto

O roteiro deste notebook segue quatro operações entre arcos: primeiro adição e subtração (que resolvem o problema de sen(15°)), depois multiplicação (arco duplo e arco triplo), divisão (arco metade) e, por fim, transformação de soma em produto.

In [2]:
sen_15_real = math.sin(math.radians(15))
tentativa_ingenua = math.sin(math.radians(45)) - math.sin(math.radians(30))

print(f"sen(15°) real:                  {sen_15_real:.6f}")
print(f"sen(45°) - sen(30°) (tentativa): {tentativa_ingenua:.6f}")
print(f"diferença: {abs(sen_15_real - tentativa_ingenua):.6f}")

assert abs(sen_15_real - tentativa_ingenua) > 1e-3, "a tentativa ingênua deveria falhar"
print("\nComo já vimos em 0505a_arcos_notaveis, subtrair senos não é a fórmula certa. Vamos deduzir a fórmula correta.")

sen(15°) real:                  0.258819
sen(45°) - sen(30°) (tentativa): 0.207107
diferença: 0.051712

Como já vimos em 0505a_arcos_notaveis, subtrair senos não é a fórmula certa. Vamos deduzir a fórmula correta.


📎 **Conexão:** este problema foi deixado em aberto em [`0505a_arcos_notaveis.ipynb`](0505a_arcos_notaveis.ipynb), Bloco 6.

**Pergunta de checagem:** além de subtração, que outras operações entre arcos poderia ser útil ter uma fórmula pronta?

## 2. Fórmulas de Adição e Subtração de Arcos

Considere dois pontos do ciclo trigonométrico, associados aos arcos a e b: P = (cos a, sen a) e Q = (cos b, sen b). A distância entre P e Q pode ser calculada de duas formas.

Pela **Lei dos Cossenos** no triângulo OPQ (O = origem, OP = OQ = 1, ângulo em O igual a a − b):

PQ² = 1² + 1² − 2·1·1·cos(a − b) = 2 − 2·cos(a − b)

Pela **fórmula de distância entre coordenadas**:

PQ² = (cos a − cos b)² + (sen a − sen b)² = 2 − 2·(cos a·cos b + sen a·sen b)

Igualando as duas expressões:

**cos(a − b) = cos a·cos b + sen a·sen b**

A partir dessa fórmula, obtemos as outras cinco:

- **cos(a + b)**: troque b por −b acima e use a paridade (cos(−b) = cos b, sen(−b) = −sen b, vistas em `0504a_reducao_ao_1o_quadrante`) → cos(a + b) = cos a·cos b − sen a·sen b
- **sen(a + b)**: use a relação de co-função sen(x) = cos(90° − x) com x = a + b, e aplique a fórmula de cos(a − b) já deduzida → sen(a + b) = sen a·cos b + cos a·sen b
- **sen(a − b)**: troque b por −b na fórmula anterior → sen(a − b) = sen a·cos b − cos a·sen b
- **tg(a + b)** e **tg(a − b)**: dividindo sen por cos em cada caso, e dividindo numerador e denominador por cos a·cos b → tg(a ± b) = (tg a ± tg b) / (1 ∓ tg a·tg b)

In [3]:
def cos_soma(a: float, b: float) -> float:
    """Calcula cos(a + b) usando a fórmula de adição (a, b em radianos)."""
    return math.cos(a) * math.cos(b) - math.sin(a) * math.sin(b)


def sen_soma(a: float, b: float) -> float:
    """Calcula sen(a + b) usando a fórmula de adição (a, b em radianos)."""
    return math.sin(a) * math.cos(b) + math.cos(a) * math.sin(b)


def cos_diferenca(a: float, b: float) -> float:
    """Calcula cos(a - b) usando a fórmula de subtração (a, b em radianos)."""
    return math.cos(a) * math.cos(b) + math.sin(a) * math.sin(b)


def sen_diferenca(a: float, b: float) -> float:
    """Calcula sen(a - b) usando a fórmula de subtração (a, b em radianos)."""
    return math.sin(a) * math.cos(b) - math.cos(a) * math.sin(b)


a_teste, b_teste = math.radians(50), math.radians(20)
assert abs(cos_soma(a_teste, b_teste) - math.cos(a_teste + b_teste)) < 1e-9
assert abs(sen_soma(a_teste, b_teste) - math.sin(a_teste + b_teste)) < 1e-9
assert abs(cos_diferenca(a_teste, b_teste) - math.cos(a_teste - b_teste)) < 1e-9
assert abs(sen_diferenca(a_teste, b_teste) - math.sin(a_teste - b_teste)) < 1e-9
print("As quatro fórmulas batem com math.cos/math.sin para a = 50° e b = 20°.")

# agora, o valor exato de sen(15°) = sen(45° - 30°), mantendo tudo simbólico com sympy
sen_45, cos_45 = sp.sqrt(2) / 2, sp.sqrt(2) / 2
sen_30, cos_30 = sp.Rational(1, 2), sp.sqrt(3) / 2

sen_15_exato = sp.simplify(sen_45 * cos_30 - cos_45 * sen_30)
cos_15_exato = sp.simplify(cos_45 * cos_30 + sen_45 * sen_30)

print(f"\nsen(15°) = sen(45° - 30°) = {sen_15_exato} ≈ {float(sen_15_exato):.6f}")
print(f"cos(15°) = cos(45° - 30°) = {cos_15_exato} ≈ {float(cos_15_exato):.6f}")

assert abs(float(sen_15_exato) - math.sin(math.radians(15))) < 1e-9
assert abs(float(cos_15_exato) - math.cos(math.radians(15))) < 1e-9
print("\nAgora sim: o valor exato de sen(15°) bate com math.sin(radians(15)).")

As quatro fórmulas batem com math.cos/math.sin para a = 50° e b = 20°.

sen(15°) = sen(45° - 30°) = -sqrt(2)/4 + sqrt(6)/4 ≈ 0.258819
cos(15°) = cos(45° - 30°) = sqrt(2)/4 + sqrt(6)/4 ≈ 0.965926

Agora sim: o valor exato de sen(15°) bate com math.sin(radians(15)).


🎯 **Aplicação prática (computação gráfica):** rotacionar um ponto (x, y) por um ângulo θ em torno da origem é uma operação constante em jogos, animações e processamento de imagens. Escrevendo o ponto como x = r·cos α, y = r·sen α (para algum ângulo α e raio r), o ponto rotacionado (x', y') está no ângulo α + θ, com o mesmo raio r:

x' = r·cos(α + θ) = r·cos α·cos θ − r·sen α·sen θ = **x·cos θ − y·sen θ**

y' = r·sen(α + θ) = r·sen α·cos θ + r·cos α·sen θ = **x·sen θ + y·cos θ**

— exatamente as fórmulas de adição aplicadas às coordenadas do ponto.

In [4]:
def rotacionar_ponto(x: float, y: float, theta_graus: float) -> tuple[float, float]:
    """Rotaciona o ponto (x, y) por um ângulo theta (em graus) em torno da origem, usando as fórmulas de adição de arcos."""
    theta = math.radians(theta_graus)
    x_novo = x * math.cos(theta) - y * math.sin(theta)
    y_novo = x * math.sin(theta) + y * math.cos(theta)
    return x_novo, y_novo


ponto_original = (2.0, 0.5)
ponto_rotacionado = rotacionar_ponto(*ponto_original, 90)
print(f"ponto original:      {ponto_original}")
print(f"rotacionado em 90°:  ({ponto_rotacionado[0]:.4f}, {ponto_rotacionado[1]:.4f})")

ponto original:      (2.0, 0.5)
rotacionado em 90°:  (-0.5000, 2.0000)


🕹️ **Widget interativo:** varie o ângulo de rotação θ e observe um triângulo girando em tempo real em torno da origem, usando as fórmulas recém-deduzidas.

In [ ]:
TRIANGULO_BASE = [(2.0, 0.5), (3.0, 0.5), (2.5, 1.5)]


def plotar_rotacao(theta_graus: float) -> None:
    pontos_rotacionados = [rotacionar_ponto(x, y, theta_graus) for x, y in TRIANGULO_BASE]

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    fig.patch.set_facecolor(NORD_FUNDO)
    estilizar_eixo(ax)

    xs_orig = [p[0] for p in TRIANGULO_BASE] + [TRIANGULO_BASE[0][0]]
    ys_orig = [p[1] for p in TRIANGULO_BASE] + [TRIANGULO_BASE[0][1]]
    ax.plot(xs_orig, ys_orig, color=NORD_LINHA, linewidth=1.5, linestyle="--", label="original")

    xs_rot = [p[0] for p in pontos_rotacionados] + [pontos_rotacionados[0][0]]
    ys_rot = [p[1] for p in pontos_rotacionados] + [pontos_rotacionados[0][1]]
    ax.plot(xs_rot, ys_rot, color=COR_PONTO, linewidth=2, label=f"rotacionado {theta_graus:.0f}°")
    ax.fill(xs_rot, ys_rot, color=COR_PONTO, alpha=0.2)

    ax.axhline(0, color=NORD_LINHA, linewidth=0.8)
    ax.axvline(0, color=NORD_LINHA, linewidth=0.8)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.legend(facecolor=NORD_PAINEL, labelcolor=NORD_TEXTO, fontsize=8, loc="upper right")
    ax.set_title(f"Rotação por θ = {theta_graus:.0f}°", color=NORD_TEXTO, fontsize=11)
    plt.show()


widgets.interact(
    plotar_rotacao,
    theta_graus=widgets.IntSlider(min=0, max=360, step=5, value=90, description="θ (°)"),
)

interactive(children=(IntSlider(value=90, description='θ (°)', max=360, step=5), Output()), _dom_classes=('wid…

<function __main__.plotar_rotacao(theta_graus: float) -> None>

📌 **Fórmulas de adição e subtração de arcos:**

| Fórmula | Expressão |
|---|---|
| cos(a − b) | cos a·cos b + sen a·sen b |
| cos(a + b) | cos a·cos b − sen a·sen b |
| sen(a + b) | sen a·cos b + cos a·sen b |
| sen(a − b) | sen a·cos b − cos a·sen b |
| tg(a + b) | (tg a + tg b) / (1 − tg a·tg b) |
| tg(a − b) | (tg a − tg b) / (1 + tg a·tg b) |

**Pergunta de checagem:** agora que temos a fórmula, qual é o valor exato de sen(15°)? E o de cos(75°)?

📖 **Leitura complementar:** [Matriz de rotação — Wikipédia](https://pt.wikipedia.org/wiki/Matriz_de_rota%C3%A7%C3%A3o)

## 3. Fórmulas de Multiplicação: Arco Duplo e Arco Triplo

Fazendo b = a nas fórmulas de adição do bloco anterior, obtemos as fórmulas de **arco duplo**:

**sen(2a) = 2·sen a·cos a**

Para cos(2a), partindo de cos a·cos a − sen a·sen a = cos²a − sen²a, e usando a relação fundamental (sen²a + cos²a = 1) para reescrever de duas formas alternativas:

**cos(2a) = cos²a − sen²a = 1 − 2·sen²a = 2·cos²a − 1**

E, dividindo sen(2a) por cos(2a):

**tg(2a) = 2·tg a / (1 − tg²a)**

O **arco triplo** é uma consequência direta: basta escrever 3a = 2a + a e aplicar novamente a fórmula de adição — a dedução completa de sen(3a) e cos(3a) fica como desafio ao final do notebook.

In [6]:
def cos_2a_v1(a: float) -> float:
    return math.cos(a) ** 2 - math.sin(a) ** 2


def cos_2a_v2(a: float) -> float:
    return 1 - 2 * math.sin(a) ** 2


def cos_2a_v3(a: float) -> float:
    return 2 * math.cos(a) ** 2 - 1


for angulo_graus in [10, 40, 73, 120]:
    a = math.radians(angulo_graus)
    v1, v2, v3 = cos_2a_v1(a), cos_2a_v2(a), cos_2a_v3(a)
    esperado = math.cos(2 * a)
    assert abs(v1 - esperado) < 1e-9
    assert abs(v2 - esperado) < 1e-9
    assert abs(v3 - esperado) < 1e-9
    print(f"a = {angulo_graus:>3}°:   cos²a-sen²a = {v1:.4f}   1-2sen²a = {v2:.4f}   2cos²a-1 = {v3:.4f}   cos(2a) = {esperado:.4f}")

print("\nAs três formas de cos(2a) sempre coincidem.")

a =  10°:   cos²a-sen²a = 0.9397   1-2sen²a = 0.9397   2cos²a-1 = 0.9397   cos(2a) = 0.9397
a =  40°:   cos²a-sen²a = 0.1736   1-2sen²a = 0.1736   2cos²a-1 = 0.1736   cos(2a) = 0.1736
a =  73°:   cos²a-sen²a = -0.8290   1-2sen²a = -0.8290   2cos²a-1 = -0.8290   cos(2a) = -0.8290
a = 120°:   cos²a-sen²a = -0.5000   1-2sen²a = -0.5000   2cos²a-1 = -0.5000   cos(2a) = -0.5000

As três formas de cos(2a) sempre coincidem.


💡 **Dica:** sempre que aparecer "arco duplo" em uma expressão, vale a pena testar as três formas de cos(2a) até achar a que simplifica melhor.

**Pergunta de checagem:** por que existem três formas diferentes e equivalentes de escrever cos(2a)?

## 4. Fórmulas de Divisão: Arco Metade

Partindo das duas formas cos(2x) = 1 − 2·sen²x e cos(2x) = 2·cos²x − 1 (bloco anterior), isolamos sen x e cos x:

sen x = ±√((1 − cos 2x)/2)          cos x = ±√((1 + cos 2x)/2)

Substituindo x = a/2 (ou seja, 2x = a), chegamos às fórmulas de **arco metade**:

**sen(a/2) = ±√((1 − cos a)/2)**          **cos(a/2) = ±√((1 + cos a)/2)**

O sinal (+ ou −) não sai automaticamente da álgebra: depende do quadrante em que a/2 está, e precisa ser decidido caso a caso antes de aplicar a fórmula.

In [7]:
cos_30_exato = sp.sqrt(3) / 2

# a/2 = 15° está no 1º quadrante, então sen(15°) e cos(15°) são positivos
sen_15_meio = sp.simplify(sp.sqrt((1 - cos_30_exato) / 2))
cos_15_meio = sp.simplify(sp.sqrt((1 + cos_30_exato) / 2))

print(f"sen(15°) via arco metade (30°/2):        {sen_15_meio} ≈ {float(sen_15_meio):.6f}")
print(f"sen(15°) via adição/subtração (Bloco 2):  {sen_15_exato} ≈ {float(sen_15_exato):.6f}")

assert abs(float(sen_15_meio) - float(sen_15_exato)) < 1e-9
assert abs(float(cos_15_meio) - float(cos_15_exato)) < 1e-9
print("\nOs dois caminhos chegam ao mesmo valor exato de sen(15°) e cos(15°).")

sen(15°) via arco metade (30°/2):        sqrt(2 - sqrt(3))/2 ≈ 0.258819
sen(15°) via adição/subtração (Bloco 2):  -sqrt(2)/4 + sqrt(6)/4 ≈ 0.258819

Os dois caminhos chegam ao mesmo valor exato de sen(15°) e cos(15°).


⚠️ **Erro comum:** esquecer de checar o quadrante de a/2 antes de escolher o sinal (+ ou −) da raiz.

**Pergunta de checagem:** por que a fórmula do arco metade tem um ± na frente, e como decidir qual sinal usar?

## 5. Tangente do Arco Metade

Dividindo sen(a/2) por cos(a/2) e manipulando algebricamente (multiplicando numerador e denominador por 2·cos(a/2), e usando as fórmulas de arco duplo em sentido inverso), chega-se a duas formas equivalentes que **não têm o problema do sinal ±**:

**tg(a/2) = sen a / (1 + cos a) = (1 − cos a) / sen a**

Essa substituição, t = tg(a/2), permite reescrever sen a e cos a inteiramente em função de t — uma ferramenta poderosa para simplificar expressões trigonométricas.

In [8]:
def tg_meio_v1(a: float) -> float:
    return math.sin(a) / (1 + math.cos(a))


def tg_meio_v2(a: float) -> float:
    return (1 - math.cos(a)) / math.sin(a)


for angulo_graus in [40, 90, 150, 200]:
    a = math.radians(angulo_graus)
    v1, v2 = tg_meio_v1(a), tg_meio_v2(a)
    esperado = math.tan(a / 2)
    assert abs(v1 - esperado) < 1e-9
    assert abs(v2 - esperado) < 1e-9
    print(f"a = {angulo_graus:>3}°:   sen a/(1+cos a) = {v1:.4f}   (1-cos a)/sen a = {v2:.4f}   tg(a/2) = {esperado:.4f}")

print("\nAs duas formas de tg(a/2) sempre coincidem — e nenhuma delas tem o problema do sinal ±.")

a =  40°:   sen a/(1+cos a) = 0.3640   (1-cos a)/sen a = 0.3640   tg(a/2) = 0.3640
a =  90°:   sen a/(1+cos a) = 1.0000   (1-cos a)/sen a = 1.0000   tg(a/2) = 1.0000
a = 150°:   sen a/(1+cos a) = 3.7321   (1-cos a)/sen a = 3.7321   tg(a/2) = 3.7321
a = 200°:   sen a/(1+cos a) = -5.6713   (1-cos a)/sen a = -5.6713   tg(a/2) = -5.6713

As duas formas de tg(a/2) sempre coincidem — e nenhuma delas tem o problema do sinal ±.


🕵️ **Curiosidade histórica:** essa substituição (conhecida como substituição de Weierstrass) transforma expressões trigonométricas em expressões racionais, uma técnica ainda usada hoje para resolver certas integrais e equações — ela vai reaparecer em `11_limite_integral_e_derivada`.

**Pergunta de checagem:** por que a fórmula tg(a/2) = sen a / (1 + cos a) não tem o problema do sinal ± que apareceu no bloco anterior?

## 6. Transformação de Soma em Produto

Somando as fórmulas já conhecidas sen(a + b) e sen(a − b):

sen(a + b) + sen(a − b) = 2·sen a·cos b

Fazendo p = a + b e q = a − b (logo a = (p + q)/2 e b = (p − q)/2), obtemos a primeira fórmula de transformação:

**sen p + sen q = 2·sen((p+q)/2)·cos((p−q)/2)**

Por um caminho análogo, chegam-se às outras três:

**sen p − sen q = 2·cos((p+q)/2)·sen((p−q)/2)**

**cos p + cos q = 2·cos((p+q)/2)·cos((p−q)/2)**

**cos p − cos q = −2·sen((p+q)/2)·sen((p−q)/2)**

In [9]:
def soma_para_produto_sen_mais(p: float, q: float) -> float:
    return 2 * math.sin((p + q) / 2) * math.cos((p - q) / 2)


def soma_para_produto_sen_menos(p: float, q: float) -> float:
    return 2 * math.cos((p + q) / 2) * math.sin((p - q) / 2)


def soma_para_produto_cos_mais(p: float, q: float) -> float:
    return 2 * math.cos((p + q) / 2) * math.cos((p - q) / 2)


def soma_para_produto_cos_menos(p: float, q: float) -> float:
    return -2 * math.sin((p + q) / 2) * math.sin((p - q) / 2)


p_teste, q_teste = math.radians(70), math.radians(20)
assert abs(soma_para_produto_sen_mais(p_teste, q_teste) - (math.sin(p_teste) + math.sin(q_teste))) < 1e-9
assert abs(soma_para_produto_sen_menos(p_teste, q_teste) - (math.sin(p_teste) - math.sin(q_teste))) < 1e-9
assert abs(soma_para_produto_cos_mais(p_teste, q_teste) - (math.cos(p_teste) + math.cos(q_teste))) < 1e-9
assert abs(soma_para_produto_cos_menos(p_teste, q_teste) - (math.cos(p_teste) - math.cos(q_teste))) < 1e-9
print("As quatro fórmulas de transformação de soma em produto batem com o cálculo direto.")

As quatro fórmulas de transformação de soma em produto batem com o cálculo direto.


🎯 **Aplicação prática (batimento sonoro):** duas ondas sonoras de frequências próximas (por exemplo, 440 Hz e 446 Hz) somadas produzem um som que "pulsa" em volume — o batimento. Usando p = 2π·f₁·t e q = 2π·f₂·t na fórmula de soma em produto:

sen(2π·f₁·t) + sen(2π·f₂·t) = 2·sen(2π·f_médio·t)·cos(2π·f_meia_dif·t)

onde f_médio = (f₁+f₂)/2 é a frequência "rápida" (praticamente igual às originais) e f_meia_dif = |f₁−f₂|/2 é a frequência "lenta" do termo em cosseno — o **envelope** que o ouvido percebe como o pulsar, numa taxa de |f₁ − f₂| pulsos por segundo.

In [10]:
def onda_com_batimento(freq1: float, freq2: float, duracao: float = 0.5, n_pontos: int = 4000):
    """Calcula a soma de duas ondas senoidais de frequências freq1 e freq2 (Hz), e o envelope (termo lento) previsto pela transformação em produto."""
    t = np.linspace(0, duracao, n_pontos)
    onda = np.sin(2 * np.pi * freq1 * t) + np.sin(2 * np.pi * freq2 * t)
    freq_meia_diferenca = abs(freq1 - freq2) / 2
    envelope = 2 * np.cos(2 * np.pi * freq_meia_diferenca * t)
    return t, onda, envelope, freq_meia_diferenca


def widget_batimento(freq1: float, freq2: float) -> None:
    t, onda, envelope, freq_meia_dif = onda_com_batimento(freq1, freq2)

    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor(NORD_FUNDO)
    estilizar_eixo(ax)

    ax.plot(t, onda, color=COR_PONTO, linewidth=0.8, label="onda somada")
    ax.plot(t, envelope, color=COR_AVISO, linewidth=1.5, linestyle="--", label="envelope (± 2·cos)")
    ax.plot(t, -envelope, color=COR_AVISO, linewidth=1.5, linestyle="--")

    ax.set_xlabel("tempo (s)", color=NORD_TEXTO_SEC)
    ax.set_ylabel("amplitude", color=NORD_TEXTO_SEC)
    ax.legend(facecolor=NORD_PAINEL, labelcolor=NORD_TEXTO, fontsize=8, loc="upper right")
    ax.set_title(
        f"f1 = {freq1:.0f} Hz, f2 = {freq2:.0f} Hz   |   batimento ≈ {2 * freq_meia_dif:.1f} Hz",
        color=NORD_TEXTO, fontsize=10,
    )
    plt.show()


widgets.interact(
    widget_batimento,
    freq1=widgets.IntSlider(min=400, max=460, step=1, value=440, description="f1 (Hz)"),
    freq2=widgets.IntSlider(min=400, max=460, step=1, value=446, description="f2 (Hz)"),
)

interactive(children=(IntSlider(value=440, description='f1 (Hz)', max=460, min=400), IntSlider(value=446, desc…

<function __main__.widget_batimento(freq1: float, freq2: float) -> None>

📌 **Fórmulas de transformação de soma em produto:**

| Fórmula | Expressão |
|---|---|
| sen p + sen q | 2·sen((p+q)/2)·cos((p−q)/2) |
| sen p − sen q | 2·cos((p+q)/2)·sen((p−q)/2) |
| cos p + cos q | 2·cos((p+q)/2)·cos((p−q)/2) |
| cos p − cos q | −2·sen((p+q)/2)·sen((p−q)/2) |

**Pergunta de checagem:** no batimento sonoro, é a soma ou a diferença das duas frequências originais que determina a velocidade do "pulsar" que ouvimos?

📖 **Leitura complementar:** [Batimentos — Wikipédia](https://pt.wikipedia.org/wiki/Batimentos)

## Resumo — cheat sheet

**Adição e subtração:**

| Fórmula | Expressão |
|---|---|
| cos(a − b) | cos a·cos b + sen a·sen b |
| cos(a + b) | cos a·cos b − sen a·sen b |
| sen(a + b) | sen a·cos b + cos a·sen b |
| sen(a − b) | sen a·cos b − cos a·sen b |
| tg(a + b) | (tg a + tg b) / (1 − tg a·tg b) |
| tg(a − b) | (tg a − tg b) / (1 + tg a·tg b) |

**Arco duplo:**
- sen(2a) = 2·sen a·cos a
- cos(2a) = cos²a − sen²a = 1 − 2·sen²a = 2·cos²a − 1
- tg(2a) = 2·tg a / (1 − tg²a)

**Arco metade** (cuidado com o sinal — depende do quadrante de a/2):
- sen(a/2) = ±√((1 − cos a)/2)
- cos(a/2) = ±√((1 + cos a)/2)
- tg(a/2) = sen a / (1 + cos a) = (1 − cos a) / sen a — sem problema de sinal

**Soma em produto:**

| Fórmula | Expressão |
|---|---|
| sen p + sen q | 2·sen((p+q)/2)·cos((p−q)/2) |
| sen p − sen q | 2·cos((p+q)/2)·sen((p−q)/2) |
| cos p + cos q | 2·cos((p+q)/2)·cos((p−q)/2) |
| cos p − cos q | −2·sen((p+q)/2)·sen((p−q)/2) |

**Quando usar cada uma:**
- Precisa de sen/cos/tg de um arco combinado (soma ou diferença de dois arcos conhecidos, como 15° ou 75°) → fórmulas de adição/subtração
- Tem um arco e precisa do dobro dele → arco duplo (ou triplo, escrevendo 3a = 2a + a)
- Tem um arco e precisa da metade dele → arco metade (ou tangente do arco metade, para evitar o sinal ±)
- Precisa transformar sen p ± sen q ou cos p ± cos q em produto, por exemplo para resolver uma equação → soma em produto

## Exercícios

**Básico**
1. Calcular sen, cos e tg de 75° usando as fórmulas de adição de arcos, conferindo com `assert`.
2. Obter sen(2a) e cos(2a) dado o valor de sen(a) e cos(a) de um ângulo qualquer.

**Intermediário**
3. Calcular sen(a/2) e cos(a/2) dado cos(a), decidindo corretamente o sinal a partir do quadrante informado.
4. Transformar uma soma de senos em produto e calcular seu valor numérico.

**Desafio**
5. Deduzir a fórmula do arco triplo (sen(3a)) a partir da fórmula de adição, escrevendo 3a como 2a + a.
6. Resolver um problema aplicado: dado duas notas musicais com frequências próximas, calcular a frequência do batimento percebido e o período desse "pulsar".

### Básico

In [ ]:
# Exercício Básico 1: calcule sen, cos e tg de 75° usando as fórmulas de
# adição de arcos (75° = 45° + 30°), com as funções sen_soma e cos_soma já
# definidas na seção 2 (divida sen por cos para obter a tangente -- não use
# math.sin/cos/tan diretamente). Guarde o resultado como um dicionário
# {"sen": ..., "cos": ..., "tg": ...}, com os valores como float
# arredondados a 4 casas decimais.
resultado_75 = ...

In [ ]:
# Verificação
a75, b75 = math.radians(45), math.radians(30)
esperado_75 = {
    "sen": round(math.sin(a75 + b75), 4),
    "cos": round(math.cos(a75 + b75), 4),
    "tg": round(math.tan(a75 + b75), 4),
}
for chave, esperado in esperado_75.items():
    obtido = resultado_75[chave]
    assert abs(obtido - esperado) < 1e-3, (
        f"Tente de novo, revise: use sen_soma(a75, b75) e cos_soma(a75, b75), "
        f"com a75 = radians(45) e b75 = radians(30), para obter {chave} "
        "(seção 2 - Fórmulas de Adição e Subtração de Arcos)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: dado sen(a) = 0.6 e cos(a) = 0.8 (um ângulo do 1º
# quadrante, triângulo 3-4-5), calcule sen(2a) e cos(2a) usando as fórmulas
# de arco duplo da seção 3 (não use math.sin/cos com o próprio ângulo a --
# use apenas sen_a_dado e cos_a_dado). Guarde o resultado como uma tupla
# (sen_2a, cos_2a), ambos arredondados a 4 casas decimais.
sen_a_dado = 0.6
cos_a_dado = 0.8
sen_2a_calculado = ...
cos_2a_calculado = ...

In [ ]:
# Verificação
esperado_sen_2a = round(2 * sen_a_dado * cos_a_dado, 4)
esperado_cos_2a = round(cos_a_dado ** 2 - sen_a_dado ** 2, 4)
assert (sen_2a_calculado, cos_2a_calculado) == (esperado_sen_2a, esperado_cos_2a), (
    "Tente de novo, revise: sen(2a) = 2·sen a·cos a e cos(2a) = cos²a - sen²a "
    "(seção 3 - Fórmulas de Multiplicação: Arco Duplo e Arco Triplo)."
)
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 3: dado cos(a) = -0.6, com a no 3º quadrante
# (180° < a < 270°), calcule sen(a/2) e cos(a/2) usando as fórmulas de arco
# metade -- decidindo corretamente o sinal de cada uma a partir do
# quadrante de a/2 (se a está entre 180° e 270°, então a/2 está entre 90° e
# 135°, ou seja, no 2º quadrante). Guarde o resultado como uma tupla
# (sen_meio, cos_meio), ambos arredondados a 4 casas decimais.
cos_a_ex3 = -0.6
sen_meio_calculado = ...
cos_meio_calculado = ...

In [ ]:
# Verificação
esperado_sen_meio = round(math.sqrt((1 - cos_a_ex3) / 2), 4)
esperado_cos_meio = round(-math.sqrt((1 + cos_a_ex3) / 2), 4)
assert (sen_meio_calculado, cos_meio_calculado) == (esperado_sen_meio, esperado_cos_meio), (
    "Tente de novo, revise: no 2º quadrante sen(a/2) = +√((1-cos a)/2) "
    "(positivo) e cos(a/2) = -√((1+cos a)/2) (negativo) "
    "(seção 4 - Fórmulas de Divisão: Arco Metade)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 4: transforme a soma sen(50°) + sen(10°) em um
# produto, usando a fórmula de soma em produto da seção 6 (com p = 50° e
# q = 10°), e calcule o valor numérico desse produto. Guarde o resultado
# (float, arredondado a 4 casas decimais) em soma_como_produto.
soma_como_produto = ...

In [ ]:
# Verificação
p4, q4 = math.radians(50), math.radians(10)
esperado_soma_produto = round(2 * math.sin((p4 + q4) / 2) * math.cos((p4 - q4) / 2), 4)
assert soma_como_produto == esperado_soma_produto, (
    "Tente de novo, revise: sen p + sen q = 2·sen((p+q)/2)·cos((p-q)/2), com "
    "p = radians(50) e q = radians(10) "
    "(seção 6 - Transformação de Soma em Produto)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 5: deduza a fórmula de sen(3a) escrevendo 3a como 2a + a
# e aplicando a fórmula de adição sen(2a + a) = sen(2a)·cos(a) + cos(2a)·sen(a),
# com sen(2a) e cos(2a) dados pelas fórmulas de arco duplo da seção 3.
# Implemente a função sen_3a(a_rad), que recebe um ângulo em radianos e
# devolve sen(3a) (float). Depois, guarde em resultados_sen_3a uma lista com
# sen_3a aplicado a math.radians(20), math.radians(50) e math.radians(100),
# arredondados a 4 casas decimais.
def sen_3a(a_rad: float) -> float:
    ...


resultados_sen_3a = ...

In [ ]:
# Verificação
def sen_3a_esperado(a_rad: float) -> float:
    sen_2a = 2 * math.sin(a_rad) * math.cos(a_rad)
    cos_2a = math.cos(a_rad) ** 2 - math.sin(a_rad) ** 2
    return sen_2a * math.cos(a_rad) + cos_2a * math.sin(a_rad)


angulos_teste = [math.radians(g) for g in (20, 50, 100)]
esperado_resultados = [round(sen_3a_esperado(a), 4) for a in angulos_teste]
assert resultados_sen_3a == esperado_resultados, (
    "Tente de novo, revise: sen(3a) = sen(2a)·cos(a) + cos(2a)·sen(a), com "
    "sen(2a) = 2·sen a·cos a e cos(2a) = cos²a - sen²a "
    "(seção 3 - Fórmulas de Multiplicação: Arco Duplo e Arco Triplo)."
)
print("Certo!")

In [ ]:
# Exercício Desafio 6: duas notas musicais têm frequências de 442 Hz e
# 437 Hz. Usando o que foi visto na seção 6 sobre batimento sonoro, calcule
# a frequência do batimento percebido (em Hz) e o período desse "pulsar"
# (em segundos, arredondado a 4 casas decimais). Guarde o resultado como
# uma tupla (freq_batimento, periodo_batimento).
freq_nota1 = 442
freq_nota2 = 437
freq_batimento_calculada = ...
periodo_batimento_calculado = ...

In [ ]:
# Verificação
esperado_freq_batimento = abs(freq_nota1 - freq_nota2)
esperado_periodo = round(1 / esperado_freq_batimento, 4)
assert (freq_batimento_calculada, periodo_batimento_calculado) == (esperado_freq_batimento, esperado_periodo), (
    "Tente de novo, revise: a frequência do batimento percebido é |f1 - f2|, "
    "e o período é o inverso dessa frequência "
    "(seção 6 - Transformação de Soma em Produto)."
)
print("Certo!")

## Glossário do bloco

| Termo | Definição |
|---|---|
| Fórmulas de adição/subtração | expressam sen, cos, tg de (a ± b) em termos de sen, cos, tg de a e b separadamente |
| Arco duplo | fórmulas para sen(2a), cos(2a), tg(2a), obtidas fazendo b = a nas fórmulas de adição |
| Arco triplo | fórmulas para sen(3a), cos(3a), obtidas escrevendo 3a = 2a + a |
| Arco metade | fórmulas para sen(a/2), cos(a/2), tg(a/2), obtidas isolando sen x e cos x em cos(2x) |
| Substituição de Weierstrass | troca t = tg(a/2), que permite escrever sen a e cos a como funções racionais de t |
| Transformação em produto | reescreve sen p ± sen q e cos p ± cos q como produtos de sen/cos de (p+q)/2 e (p−q)/2 |
| Batimento sonoro | pulsar de volume percebido quando duas ondas sonoras de frequências próximas são somadas |

## Conexões

**Isso depende de:**
- [`0502a_funcoes_circulares.ipynb`](0502a_funcoes_circulares.ipynb) (definição das seis funções circulares)
- [`0503a_relacoes_fundamentais.ipynb`](0503a_relacoes_fundamentais.ipynb) (relação fundamental, usada na dedução do arco duplo e do arco metade)
- [`0504a_reducao_ao_1o_quadrante.ipynb`](0504a_reducao_ao_1o_quadrante.ipynb) (paridade das funções circulares, usada na dedução das fórmulas de subtração)
- [`0505a_arcos_notaveis.ipynb`](0505a_arcos_notaveis.ipynb) (este notebook resolve a pergunta em aberto deixada ali)

**Isso será usado em:**
- [`0507a_equacoes.ipynb`](0507a_equacoes.ipynb) (equações trigonométricas mais elaboradas frequentemente exigem transformar a expressão usando as fórmulas deste notebook antes de resolver)
- [`0510a_triangulos_quaisquer.ipynb`](0510a_triangulos_quaisquer.ipynb) (a Lei dos Cossenos e certas deduções da Lei dos Senos se apoiam nas fórmulas de adição de arcos)
- [`11_limite_integral_e_derivada/`](../11_limite_integral_e_derivada/) (as fórmulas de arco metade e a substituição de Weierstrass reaparecem no cálculo de integrais de expressões trigonométricas)